In [1]:
!pip install pyspark
!wget -O owid-covid-data.csv https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# -----------------------------
# 1. Spark Session
# -----------------------------
spark = SparkSession.builder.appName("COVID19 Analysis").getOrCreate()

# -----------------------------
# 2. Load Dataset
# -----------------------------
df = spark.read.csv("owid-covid-data.csv", header=True, inferSchema=True)

# -----------------------------
# 3. Clean Data
# -----------------------------
df_clean = df.select(
    "location",
    "date",
    col("total_cases").cast("double"),
    col("total_deaths").cast("double")
)

# Remove NULL + zero values
df_filtered = df_clean.filter(
    (col("location").isNotNull()) &
    (col("total_cases").isNotNull()) &
    (col("total_deaths").isNotNull()) &
    (col("total_cases") > 0) &
    (col("total_deaths") > 0)
)

print("\nCLEAN DATA LOADED\n")

# -----------------------------
# 4. Show sample clean data
# -----------------------------
df_filtered.show(5)

# -----------------------------
# 5. Pakistan data (no zero values)
# -----------------------------
print("\nPakistan COVID Data:\n")

df_filtered.filter(col("location") == "Pakistan") \
    .select("date", "total_cases", "total_deaths") \
    .orderBy(col("date")) \
    .show(10)

# -----------------------------
# 6. Top countries by cases
# -----------------------------
print("\nTop 10 Countries by Cases:\n")

df_filtered.groupBy("location") \
    .max("total_cases") \
    .orderBy(col("max(total_cases)").desc()) \
    .show(10)

# -----------------------------
# 7. Top countries by deaths
# -----------------------------
print("\nTop 10 Countries by Deaths:\n")

df_filtered.groupBy("location") \
    .max("total_deaths") \
    .orderBy(col("max(total_deaths)").desc()) \
    .show(10)

# -----------------------------
# 8. Latest data (most recent records)
# -----------------------------
print("\nLatest COVID Records:\n")

df_filtered.orderBy(col("date").desc()).show(10)

# -----------------------------
# 9. Insights
# -----------------------------
print("\nINSIGHTS:")
print("1. USA, India, Brazil have highest COVID-19 cases globally.")
print("2. Pakistan shows lower but consistent case numbers.")
print("3. Death counts vary across countries.")
print("4. Zero values removed for meaningful analysis.")

--2026-05-19 21:32:23--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘owid-covid-data.csv’

owid-covid-data.csv 100%[===================>]  93.83M   200MB/s    in 0.5s    

2026-05-19 21:32:24 (200 MB/s) - ‘owid-covid-data.csv’ saved [98391483/98391483]

